[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/21.%20The%20Yard%20Block%20Housekeeping%20Problem/P21-Tier-4_executed.ipynb)

---


# 21. The Yard Block Housekeeping Problem

## Tier 4: Deep Reinforcement Learning

### Goal and workflow

This notebook formulates yard housekeeping as a sequential decision problem and trains a Deep Q-Network (DQN) agent to choose container moves. The workflow follows the reinforcement-learning approach used in this tier: define a simulated environment, represent its state and feasible actions, calculate rewards, train a dueling DQN with experience replay and a target network, and inspect the learned policy and its performance.

The inputs are a simulated 10-block yard, container attributes, movement costs, reward weights, and DQN training settings. The outputs are training statistics, evaluation summaries, plots, policy-analysis summaries, and scenario evaluations. Training outcomes are produced by running the code; they are not predetermined by this notebook.

### Scope and modeling assumptions

- A container's weight is measured in TEU and is either 2 or 4. A block's load and capacity are also measured in TEU. Every simulated block load is therefore an even number of TEU, and each permitted move transfers an exact total of 4, 8, or 12 TEU.
- An action specifies a source block, destination block, and quantity. A move is feasible only when a subset of containers with exactly that total weight can be transferred, the source is above its target utilization, and the destination is below its target utilization with sufficient capacity. A no-op action is always available.
- The reward combines utilization improvement, movement cost, accessibility, destination congestion, and urgency. Its weights express the relative importance and sign of these components; they do not guarantee that every reward component improves on every move.
- The episode ends when all blocks are within the stated utilization tolerance or when the time-step limit is reached. Because total yard load is conserved, the target utilization is not necessarily attainable for every generated initial yard.
- This is a small illustrative simulator, not a complete terminal operating model. It does not model stacking geometry, retrieval operations, equipment scheduling, stochastic arrivals during an episode, or uncertainty in travel times. Demand, weather, and equipment attributes are included in the agent's observation, but they do not all alter transition feasibility or physical movement costs. Scenario evaluations should be interpreted within these limits.

### Required background: a Markov decision process

A Markov decision process (MDP) describes repeated decisions. At time \(t\), the environment has state \(s_t\); the agent chooses action \(a_t\); the environment returns reward \(r_t\) and next state \(s_{t+1}\). The Markov assumption means that the current state contains the information needed to model the next-state and reward distributions:

$$
P(s_{t+1}, r_t \mid s_t, a_t, s_{t-1}, a_{t-1}, \ldots)
= P(s_{t+1}, r_t \mid s_t, a_t).
$$

The full simulator state contains the blocks, containers, and time-dependent information and determines the transition. The agent receives a 48-dimensional feature vector, which records selected yard features rather than every detail of that simulator state. Consequently, different simulator states can produce the same observation while having different feasible actions or rewards. The agent therefore learns from an approximate, potentially partially observed representation; the Markov property of the full simulator state does not automatically hold for this feature vector.

### Deep Q-learning and the dueling network

The action-value function \(Q(s,a)\) estimates the expected discounted return from taking action \(a\) in state \(s\) and then following the learned policy. With discount factor \(\gamma\), its Bellman optimality relation is

$$
Q^*(s,a) = \mathbb{E}\left[r + \gamma \max_{a' \in A(s')} Q^*(s',a') \mid s,a\right],
$$

where \(A(s')\) is the set of feasible actions in the next state. DQN approximates \(Q(s,a)\) with a neural network and learns from sampled transitions. For a transition with reward \(r\), next state \(s'\), and terminal indicator \(d\), the target used here is

$$
y = r + \gamma (1-d) \max_{a' \in A(s')} Q_{\text{target}}(s',a').
$$

For a terminal transition, \(d=1\), so the target is just the immediate reward; future value is not added. For a nonterminal transition, the target includes the discounted value of the best feasible next action. In code, `DQNAgent.train` builds this target, masks infeasible next actions, computes the loss, and updates the online network. Experience replay samples past transitions for learning; the target network supplies a more slowly changing target.

The dueling architecture estimates a state value \(V(s)\) and action advantages \(A(s,a)\) separately, then combines them:

$$
Q(s,a) = V(s) + \left(A(s,a) - \frac{1}{|\mathcal{A}|}\sum_{a' \in \mathcal{A}} A(s,a')\right).
$$

Here, \(\mathcal{A}\) is the full discrete action set. Centering the advantages makes the decomposition identifiable: adding the same constant to every advantage does not arbitrarily shift the action values. The code implements this combination in `YardHousekeepingDQN.forward`.

During training, epsilon-greedy action selection explores by choosing a feasible action at random with probability \(\epsilon\), and otherwise chooses the feasible action with the largest estimated Q-value. The discount factor, exploration schedule, network size, replay capacity, and target-update frequency are specified in the code. They are training settings, not evidence that the learned policy is optimal or has converged.

### What to examine

- Episode rewards and their moving average describe observed training performance; they do not by themselves prove convergence.
- Epsilon shows how the exploration probability changes during training.
- Training loss and Q-value summaries help diagnose the optimization process, but low loss alone does not establish good policy performance.
- Evaluation and baseline results are empirical summaries for this simulator and its generated yard. They should not be generalized to real terminal operations without additional validation.
- Policy and scenario analyses describe the actions and conditions actually evaluated by the code. They do not establish emergent behavior, robustness, or superiority in advance.

In [ ]:
from collections import defaultdict, deque
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import copy
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    # Small replay minibatches make frequent thread-pool startup costly on CPU.
    torch.set_num_threads(1)

print(f"Using device: {device}")

## 1. Represent the yard

The environment needs explicit data structures so that state features, actions, and physical quantities have consistent meanings. Container weight, block load, capacity, and action quantity are all measured in TEU. A `YardState` also maintains a block-indexed container collection; this makes it possible to identify the exact containers transferred by a move.

In [ ]:
@dataclass
class Container:
    """Container attributes used by the simulated environment."""

    id: int
    block_id: int
    urgency: float
    accessibility: float
    weight: float
    time_until_retrieval: int
    value: float


@dataclass
class Block:
    """A yard block with load, capacity, and operating attributes."""

    id: int
    current_load: int
    capacity: int
    target_utilization: float
    equipment_availability: float
    distance_from_berth: float
    congestion_level: float
    maintenance_scheduled: bool

    @property
    def utilization(self) -> float:
        return self.current_load / self.capacity

    @property
    def available_capacity(self) -> int:
        return self.capacity - self.current_load

    @property
    def utilization_gap(self) -> float:
        return self.utilization - self.target_utilization


@dataclass
class Action:
    """A discrete move between blocks, or the no-op action."""

    source_block: int
    destination_block: int
    quantity: int
    action_id: int

    def __str__(self) -> str:
        if self.quantity == 0:
            return "No-op"
        return (
            f"Move {self.quantity} TEU: Block {self.source_block} "
            f"to Block {self.destination_block}"
        )


@dataclass
class RLReward:
    """Unweighted reward components and their weighted total."""

    utilization_improvement: float
    movement_cost: float
    accessibility_bonus: float
    congestion_penalty: float
    urgency_satisfaction: float
    total: float


@dataclass
class YardState:
    """Yard configuration and time-dependent state information."""

    blocks: List[Block]
    containers: List[Container]
    movement_costs: np.ndarray
    time_step: int = 0
    demand_forecast: Optional[List[float]] = None
    weather_conditions: float = 1.0

    def __post_init__(self) -> None:
        self.num_blocks = len(self.blocks)
        self.containers_by_block = defaultdict(list)
        for container in self.containers:
            self.containers_by_block[container.block_id].append(container)
        if self.demand_forecast is None:
            self.demand_forecast = [0.8] * 24

    def get_block_by_id(self, block_id: int) -> Block:
        return next(block for block in self.blocks if block.id == block_id)

    def get_containers_in_block(self, block_id: int) -> List[Container]:
        return self.containers_by_block[block_id]

    def copy(self) -> "YardState":
        return copy.deepcopy(self)

## 2. Generate a simulated yard

The generator creates ten blocks, container attributes, a demand forecast, and a directed movement-cost matrix. The sampled load is rounded down to an even number of TEU because all generated containers weigh either 2 or 4 TEU, and it is capped at the block's physical capacity. Container weights are then generated to sum exactly to each block's load. This makes the initial block loads agree with the containers represented in the state while keeping every block within capacity.

In [ ]:
def create_rl_environment() -> YardState:
    """Create a 10-block yard with internally consistent TEU loads."""
    blocks = []
    containers = []
    container_id = 1

    for i in range(10):
        sampled_load = np.random.randint(70, 130)
        capacity = 120
        current_load = min(sampled_load - sampled_load % 2, capacity)
        target_utilization = 0.80

        block = Block(
            id=i + 1,
            current_load=current_load,
            capacity=capacity,
            target_utilization=target_utilization,
            equipment_availability=np.random.uniform(0.6, 1.0),
            distance_from_berth=float(i),
            congestion_level=np.random.uniform(0.0, 0.8),
            maintenance_scheduled=np.random.random() < 0.1,
        )
        blocks.append(block)

        remaining_load = current_load
        while remaining_load > 0:
            choose_four_teu = remaining_load >= 4 and np.random.random() >= 0.7
            weight = 4 if choose_four_teu else 2
            remaining_load -= weight
            containers.append(
                Container(
                    id=container_id,
                    block_id=block.id,
                    urgency=np.random.beta(2, 2),
                    accessibility=np.random.beta(3, 2),
                    weight=weight,
                    time_until_retrieval=np.random.randint(1, 25),
                    value=np.random.uniform(0.5, 1.0),
                )
            )
            container_id += 1

    movement_costs = np.zeros((10, 10))
    for i in range(10):
        for j in range(10):
            if i != j:
                base_cost = 10 + abs(i - j) * 3
                movement_costs[i, j] = max(
                    5, base_cost + np.random.uniform(-3, 3)
                )

    yard_state = YardState(
        blocks=blocks,
        containers=containers,
        movement_costs=movement_costs,
        time_step=0,
        demand_forecast=[
            0.8 + 0.1 * np.sin(i / 4) for i in range(24)
        ],
        weather_conditions=1.0,
    )

    for block in yard_state.blocks:
        container_load = sum(
            container.weight
            for container in yard_state.get_containers_in_block(block.id)
        )
        assert container_load == block.current_load
        assert block.current_load <= block.capacity

    return yard_state


yard_env = create_rl_environment()
utilizations = [block.utilization for block in yard_env.blocks]

print("10-block simulated yard")
print(f"Blocks: {len(yard_env.blocks)}")
print(f"Containers: {len(yard_env.containers)}")
print(f"Total load: {sum(block.current_load for block in yard_env.blocks)} TEU")
print(f"Mean block utilization: {np.mean(utilizations):.1%}")
print(f"Utilization standard deviation: {np.std(utilizations):.1%}")
print(f"Minimum block utilization: {np.min(utilizations):.1%}")
print(f"Maximum block utilization: {np.max(utilizations):.1%}")

## 3. Define the reinforcement-learning environment

A state is the information supplied to the agent; an action is one candidate decision; a transition applies that decision and returns a new encoded state and reward. The action set contains every ordered pair of distinct blocks at move quantities of 4, 8, or 12 TEU, plus a no-op. With \(B\) blocks, its fixed size is \(3B(B-1)+1\). For the ten-block example, the action space therefore contains all 270 directed move choices and one no-op. The set is fixed, but only actions feasible in the current state are offered to the policy.

For a state \(s\), the feasible action set \(A(s)\) includes the no-op and a move only if the source can supply a matching subset of containers, the source is above its target utilization, and the destination is below its target utilization with enough free capacity. The subset check is a small subset-sum search: it builds reachable totals from the source containers and retains one container subset for each reachable total up to the requested quantity. This matters because a block load alone does not establish that an exact action quantity can be formed from whole containers. For efficiency, the code reuses each source-and-quantity subset result across candidate destinations during a feasibility check. If a source contains \(n\) containers and the requested quantity is \(q\), the bounded subset-sum search takes \(O(nq)\) time and stores reachable totals only up to \(q\); here, \(q\) is at most 12 TEU.

A move transfers the selected whole containers, reduces the source load by the action quantity, and increases the destination load by the same amount. Thus total yard load is conserved. Feasibility is assessed before the move; a move can cross a target utilization threshold because the action quantities are discrete.

The reward is a weighted sum. If \(I(s,a,s')\) is the mean reduction in absolute utilization gap, \(C(a)\) is normalized movement cost, \(A(s,a)\) is the accessibility bonus, \(G(s,a)\) is the congestion penalty, and \(U(s,a)\) is the urgency score, then

$$
r(s,a,s') = 0.4I(s,a,s') - 0.3C(a) + 0.2A(s,a) - 0.1G(s,a) + 0.4U(s,a).
$$

In the code, \(I\) is the average across blocks of the decrease in absolute utilization gap. For a move of quantity \(q\) from block \(i\) to block \(j\), \(C(a)\) is the directed movement cost from \(i\) to \(j\), multiplied by \(q/1000\). The accessibility term is 0.1 times the mean accessibility of the moved containers. The congestion term is 0.1 times the destination's congestion level, and the urgency term is 0.1 times the number of moved containers with urgency above 0.7 divided by the number of containers in the source block before the move. For a no-op, all move-specific terms are zero. The component definitions and their implementation are in `calculate_reward`. A positive value of a component such as `movement_cost` means that the cost was incurred; its negative weight makes it reduce the total reward.

In [ ]:
class YardHousekeepingEnvironment:
    """Finite-horizon yard environment for DQN training and evaluation."""

    def __init__(self, yard_state: YardState):
        self.yard_state = yard_state
        self.initial_state = yard_state.copy()
        self.current_state = yard_state.copy()
        self.time_step = 0
        self.max_time_steps = 50
        self.action_space = self._create_action_space()
        self.state_dim = self._get_state_dimension()
        self.reward_weights = {
            "utilization_improvement": 0.4,
            "movement_cost": -0.3,
            "accessibility_bonus": 0.2,
            "congestion_penalty": -0.1,
            "urgency_satisfaction": 0.4,
        }
        self._valid_action_cache = {}

    def _create_action_space(self) -> List[Action]:
        """Enumerate ordered block moves and append a no-op action."""
        actions = []
        action_id = 0
        for source in range(1, self.yard_state.num_blocks + 1):
            for destination in range(1, self.yard_state.num_blocks + 1):
                if source == destination:
                    continue
                for quantity in (4, 8, 12):
                    actions.append(
                        Action(source, destination, quantity, action_id)
                    )
                    action_id += 1

        actions.append(Action(0, 0, 0, action_id))
        return actions

    def _get_state_dimension(self) -> int:
        """Return the feature count for a 10-block state vector."""
        return 10 + 10 + 10 + 1 + 1 + 6 + 10

    def encode_state(self, state: YardState) -> np.ndarray:
        """Encode selected block, weather, time, and demand features."""
        features = []
        features.extend(block.utilization for block in state.blocks)
        features.extend(block.congestion_level for block in state.blocks)
        features.extend(
            block.equipment_availability for block in state.blocks
        )
        features.append(state.weather_conditions)
        features.append(state.time_step / self.max_time_steps)

        forecast_start = min(
            state.time_step, max(0, len(state.demand_forecast) - 6)
        )
        for i in range(6):
            forecast_index = forecast_start + i
            if forecast_index < len(state.demand_forecast):
                features.append(state.demand_forecast[forecast_index])
            else:
                features.append(0.8)

        for block in state.blocks:
            containers = state.get_containers_in_block(block.id)
            average_urgency = (
                np.mean([container.urgency for container in containers])
                if containers
                else 0.0
            )
            features.append(average_urgency)

        encoded = np.asarray(features, dtype=np.float32)
        if encoded.shape != (self.state_dim,):
            raise ValueError("Encoded state has an unexpected dimension.")
        return encoded

    @staticmethod
    def _select_containers_for_quantity(
        state: YardState, source_block_id: int, quantity: int
    ) -> Optional[List[Container]]:
        """Find a deterministic subset whose total weight equals quantity."""
        containers = state.get_containers_in_block(source_block_id)
        reachable = {0: []}
        for container in containers:
            weight = int(container.weight)
            for subtotal, selected in list(reachable.items()):
                new_total = subtotal + weight
                if new_total <= quantity and new_total not in reachable:
                    reachable[new_total] = selected + [container]
            if quantity in reachable:
                return reachable[quantity]
        return reachable.get(quantity)

    def _valid_action_cache_key(self, state: YardState) -> Tuple:
        """Summarize every state feature that determines move feasibility."""
        container_counts = {
            block.id: [0, 0] for block in state.blocks
        }
        for container in state.containers:
            weight = int(container.weight)
            if weight not in (2, 4):
                raise ValueError("Container weights must be 2 or 4 TEU.")
            container_counts[container.block_id][weight // 2 - 1] += 1

        return tuple(
            (
                block.id,
                block.current_load,
                block.capacity,
                block.target_utilization,
                tuple(container_counts[block.id]),
            )
            for block in state.blocks
        )

    def get_valid_actions(self, state: YardState) -> List[int]:
        """Return action indices feasible in the supplied state."""
        cache_key = self._valid_action_cache_key(state)
        cached_actions = self._valid_action_cache.get(cache_key)
        if cached_actions is not None:
            return list(cached_actions)

        valid_actions = []
        no_op_index = len(self.action_space) - 1
        subset_cache = {}

        for index, action in enumerate(self.action_space):
            if index == no_op_index:
                valid_actions.append(index)
                continue

            source = state.get_block_by_id(action.source_block)
            destination = state.get_block_by_id(action.destination_block)
            if not (
                source.current_load >= action.quantity
                and destination.available_capacity >= action.quantity
                and source.utilization > source.target_utilization
                and destination.utilization < destination.target_utilization
            ):
                continue

            subset_key = (action.source_block, action.quantity)
            if subset_key not in subset_cache:
                subset_cache[subset_key] = self._select_containers_for_quantity(
                    state, action.source_block, action.quantity
                )
            if subset_cache[subset_key] is not None:
                valid_actions.append(index)

        self._valid_action_cache[cache_key] = tuple(valid_actions)
        if len(self._valid_action_cache) > 4096:
            self._valid_action_cache.pop(next(iter(self._valid_action_cache)))
        return valid_actions

    def calculate_reward(
        self,
        state: YardState,
        action: Action,
        next_state: YardState,
        moved_containers: Optional[List[Container]] = None,
    ) -> RLReward:
        """Calculate the weighted reward and retain its components."""
        utilization_improvement = np.mean(
            [
                abs(block.utilization_gap)
                - abs(
                    next_state.get_block_by_id(block.id).utilization_gap
                )
                for block in state.blocks
            ]
        )

        if action.quantity > 0:
            movement_cost = (
                state.movement_costs[
                    action.source_block - 1,
                    action.destination_block - 1,
                ]
                * action.quantity
                / 1000.0
            )
            moved_containers = moved_containers or []
            accessibility_bonus = (
                float(np.mean([item.accessibility for item in moved_containers]))
                * 0.1
                if moved_containers
                else 0.0
            )
            destination = state.get_block_by_id(action.destination_block)
            congestion_penalty = destination.congestion_level * 0.1
            source_containers = state.get_containers_in_block(action.source_block)
            urgency_satisfaction = (
                sum(item.urgency > 0.7 for item in moved_containers)
                / len(source_containers)
                * 0.1
                if source_containers
                else 0.0
            )
        else:
            movement_cost = 0.0
            accessibility_bonus = 0.0
            congestion_penalty = 0.0
            urgency_satisfaction = 0.0

        total = (
            self.reward_weights["utilization_improvement"]
            * utilization_improvement
            + self.reward_weights["movement_cost"] * movement_cost
            + self.reward_weights["accessibility_bonus"]
            * accessibility_bonus
            + self.reward_weights["congestion_penalty"]
            * congestion_penalty
            + self.reward_weights["urgency_satisfaction"]
            * urgency_satisfaction
        )

        return RLReward(
            utilization_improvement=float(utilization_improvement),
            movement_cost=float(movement_cost),
            accessibility_bonus=float(accessibility_bonus),
            congestion_penalty=float(congestion_penalty),
            urgency_satisfaction=float(urgency_satisfaction),
            total=float(total),
        )

    def step(self, action_index: int) -> Tuple[np.ndarray, float, bool, Dict]:
        """Apply one feasible action and return state, reward, and status."""
        if not 0 <= action_index < len(self.action_space):
            raise IndexError("Action index is outside the action space.")

        valid_actions = self.get_valid_actions(self.current_state)
        if action_index not in valid_actions:
            raise ValueError("The selected action is not feasible in this state.")

        action = self.action_space[action_index]
        previous_state = copy.copy(self.current_state)
        previous_state.blocks = [copy.copy(block) for block in self.current_state.blocks]
        previous_state.containers_by_block = defaultdict(
            list,
            {
                block_id: list(containers)
                for block_id, containers
                in self.current_state.containers_by_block.items()
            },
        )
        moved_containers = []

        if action.quantity > 0:
            moved_containers = self._select_containers_for_quantity(
                self.current_state, action.source_block, action.quantity
            )
            if moved_containers is None:
                raise RuntimeError("A feasible move has no matching container subset.")

            source = self.current_state.get_block_by_id(action.source_block)
            destination = self.current_state.get_block_by_id(
                action.destination_block
            )
            source.current_load -= action.quantity
            destination.current_load += action.quantity

            for container in moved_containers:
                self.current_state.containers_by_block[action.source_block].remove(
                    container
                )
                container.block_id = action.destination_block
                self.current_state.containers_by_block[
                    action.destination_block
                ].append(container)

        self.time_step += 1
        self.current_state.time_step = self.time_step
        reward = self.calculate_reward(
            previous_state, action, self.current_state, moved_containers
        )
        done = (
            self.time_step >= self.max_time_steps
            or self._is_terminal_state()
        )
        next_state = self.encode_state(self.current_state)
        utilization_variance = float(
            np.var([block.utilization for block in self.current_state.blocks])
        )
        info = {
            "reward_components": reward,
            "action_taken": action,
            "utilization_variance": utilization_variance,
        }
        return next_state, reward.total, done, info

    def _is_terminal_state(self) -> bool:
        """Check whether every block is within five percentage points."""
        return all(
            abs(block.utilization_gap) <= 0.05
            for block in self.current_state.blocks
        )

    def reset(self) -> np.ndarray:
        """Restore the initial yard and return its encoded state."""
        self.current_state = self.initial_state.copy()
        self.time_step = 0
        self.current_state.time_step = 0
        return self.encode_state(self.current_state)

    def render(self) -> None:
        """Print block utilization and the current utilization variance."""
        print(f"Time step: {self.time_step}")
        for block in self.current_state.blocks:
            print(
                f"Block {block.id}: {block.utilization:.1%} "
                f"(target: {block.target_utilization:.1%})"
            )
        variance = np.var(
            [block.utilization for block in self.current_state.blocks]
        )
        print(f"Utilization variance: {variance:.4f}")


env = YardHousekeepingEnvironment(yard_env)
initial_state = env.reset()
valid_actions = env.get_valid_actions(env.current_state)
assert initial_state.shape == (env.state_dim,)
assert len(valid_actions) > 0

print(f"State dimension: {env.state_dim}")
print(f"Action-space size: {len(env.action_space)}")
print(f"Initially feasible actions, including no-op: {len(valid_actions)}")
print(f"Maximum episode length: {env.max_time_steps} steps")

## 4. Build the dueling DQN and learning agent

The network maps one encoded state to one Q-value per action. Its shared feature layers feed a scalar value head and a vector advantage head. The agent uses epsilon-greedy selection, a replay buffer, a target network, and gradient clipping. Infeasible actions are masked both when selecting actions and when calculating the next-state maximum in the Bellman target; without the second mask, an infeasible action could incorrectly influence learning.

In [ ]:
class YardHousekeepingDQN(nn.Module):
    """Dueling neural network that estimates values for yard actions."""

    def __init__(
        self, state_dim: int, action_dim: int, hidden_dim: int = 256
    ):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.value_head = nn.Sequential(
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU(),
            nn.Linear(hidden_dim // 4, 1),
        )
        self.advantage_head = nn.Sequential(
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU(),
            nn.Linear(hidden_dim // 4, action_dim),
        )

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        features = self.feature_extractor(state)
        value = self.value_head(features)
        advantage = self.advantage_head(features)
        return value + advantage - advantage.mean(dim=1, keepdim=True)


class DQNAgent:
    """DQN learner with replay, a target network, and action masking."""

    def __init__(
        self, state_dim: int, action_dim: int, learning_rate: float = 0.001
    ):
        self.device = device
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.q_network = YardHousekeepingDQN(
            state_dim, action_dim
        ).to(self.device)
        self.target_network = YardHousekeepingDQN(
            state_dim, action_dim
        ).to(self.device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()
        self.optimizer = optim.Adam(
            self.q_network.parameters(), lr=learning_rate
        )

        self.memory = deque(maxlen=10000)
        self.batch_size = 64
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.gamma = 0.95
        self.target_update_freq = 100
        self.training_step = 0
        self.losses = []
        self.q_values = []

    def select_action(
        self, state: np.ndarray, valid_actions: List[int]
    ) -> int:
        """Choose a feasible action using epsilon-greedy exploration."""
        if not valid_actions:
            raise ValueError("At least one feasible action is required.")
        if random.random() < self.epsilon:
            return random.choice(valid_actions)

        was_training = self.q_network.training
        self.q_network.eval()
        state_tensor = torch.as_tensor(
            state, dtype=torch.float32, device=self.device
        ).unsqueeze(0)
        with torch.no_grad():
            q_values = self.q_network(state_tensor).squeeze(0)
        if was_training:
            self.q_network.train()

        valid_tensor = torch.as_tensor(
            valid_actions, dtype=torch.long, device=self.device
        )
        best_position = torch.argmax(q_values[valid_tensor]).item()
        return int(valid_actions[best_position])

    def store_experience(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        done: bool,
        next_valid_actions: List[int],
    ) -> None:
        """Store a transition together with its feasible next actions."""
        next_action_mask = np.zeros(self.action_dim, dtype=np.bool_)
        next_action_mask[next_valid_actions] = True
        self.memory.append(
            (
                state.copy(),
                action,
                reward,
                next_state.copy(),
                done,
                next_action_mask,
            )
        )

    def train(self) -> Optional[float]:
        """Update the online network from a replay-buffer minibatch."""
        if len(self.memory) < self.batch_size:
            return None

        batch = random.sample(self.memory, self.batch_size)
        states, actions, rewards, next_states, dones, next_masks = zip(*batch)
        states_tensor = torch.as_tensor(
            np.stack(states), dtype=torch.float32, device=self.device
        )
        actions_tensor = torch.as_tensor(
            actions, dtype=torch.long, device=self.device
        )
        rewards_tensor = torch.as_tensor(
            rewards, dtype=torch.float32, device=self.device
        )
        next_states_tensor = torch.as_tensor(
            np.stack(next_states), dtype=torch.float32, device=self.device
        )
        dones_tensor = torch.as_tensor(
            dones, dtype=torch.float32, device=self.device
        )
        next_masks_tensor = torch.as_tensor(
            np.stack(next_masks), dtype=torch.bool, device=self.device
        )

        current_q = self.q_network(states_tensor).gather(
            1, actions_tensor.unsqueeze(1)
        ).squeeze(1)

        with torch.no_grad():
            next_q_values = self.target_network(next_states_tensor)
            next_q_values = next_q_values.masked_fill(
                ~next_masks_tensor, float("-inf")
            )
            max_next_q = next_q_values.max(dim=1).values
            targets = rewards_tensor + self.gamma * (1 - dones_tensor) * max_next_q

        loss = F.mse_loss(current_q, targets)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), 1.0)
        self.optimizer.step()

        self.training_step += 1
        if self.training_step % self.target_update_freq == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())
            self.target_network.eval()

        self.epsilon = max(
            self.epsilon_min, self.epsilon * self.epsilon_decay
        )
        self.losses.append(float(loss.item()))
        self.q_values.append(float(current_q.detach().mean().item()))
        return float(loss.item())

    def evaluate(
        self, env: YardHousekeepingEnvironment, num_episodes: int = 10
    ) -> Dict:
        """Measure greedy policy performance on repeated episodes."""
        previous_epsilon = self.epsilon
        self.epsilon = 0.0
        total_rewards = []
        total_steps = []
        final_variances = []

        try:
            for _ in range(num_episodes):
                state = env.reset()
                episode_reward = 0.0
                steps = 0
                done = False
                info = {}

                while not done and steps < env.max_time_steps:
                    valid_actions = env.get_valid_actions(env.current_state)
                    action = self.select_action(state, valid_actions)
                    next_state, reward, done, info = env.step(action)
                    episode_reward += reward
                    steps += 1
                    state = next_state

                total_rewards.append(episode_reward)
                total_steps.append(steps)
                final_variances.append(info["utilization_variance"])
        finally:
            self.epsilon = previous_epsilon

        return {
            "avg_reward": float(np.mean(total_rewards)),
            "std_reward": float(np.std(total_rewards)),
            "avg_steps": float(np.mean(total_steps)),
            "avg_final_variance": float(np.mean(final_variances)),
            "total_rewards": total_rewards,
        }


agent = DQNAgent(env.state_dim, len(env.action_space))
parameter_count = sum(
    parameter.numel() for parameter in agent.q_network.parameters()
)
print(f"State dimension: {agent.state_dim}")
print(f"Action dimension: {agent.action_dim}")
print(f"Compute device: {agent.device}")
print(f"Initial exploration probability: {agent.epsilon:.3f}")
print(f"Trainable network parameters: {parameter_count:,}")

## 5. Train the agent

Training repeatedly resets the same generated initial yard. In each episode, the agent selects a feasible action, observes a transition, stores the transition and feasible next-action mask, and updates the network once the replay buffer contains a full minibatch. The target network is synchronized at the specified training-step interval. Periodic evaluations use a greedy policy and restore the training exploration probability afterward.

The Bellman target uses a terminal indicator so that no future value is added after an episode ends. The maximum is taken only over feasible next actions. The number of episodes and evaluation frequency below are the experiment settings; the notebook does not assume a particular reward, convergence point, or comparison outcome.

In [ ]:
def train_agent(
    agent: DQNAgent,
    env: YardHousekeepingEnvironment,
    num_episodes: int = 1000,
    eval_freq: int = 100,
) -> Dict:
    """Train the DQN agent and record episode-level statistics."""
    if num_episodes <= 0 or eval_freq <= 0:
        raise ValueError("Episode count and evaluation frequency must be positive.")

    episode_rewards = []
    episode_lengths = []
    evaluation_results = []
    epsilon_history = []

    print("=" * 72)
    print("DEEP REINFORCEMENT LEARNING TRAINING")
    print("=" * 72)

    for episode in range(num_episodes):
        state = env.reset()
        episode_reward = 0.0
        episode_length = 0
        done = False

        while not done and episode_length < env.max_time_steps:
            valid_actions = env.get_valid_actions(env.current_state)
            action = agent.select_action(state, valid_actions)
            next_state, reward, done, _ = env.step(action)
            next_valid_actions = env.get_valid_actions(env.current_state)
            agent.store_experience(
                state,
                action,
                reward,
                next_state,
                done,
                next_valid_actions,
            )
            agent.train()
            episode_reward += reward
            episode_length += 1
            state = next_state

        episode_rewards.append(episode_reward)
        episode_lengths.append(episode_length)
        epsilon_history.append(agent.epsilon)

        if (episode + 1) % eval_freq == 0:
            evaluation = agent.evaluate(env, num_episodes=5)
            evaluation_results.append(evaluation)
            recent_average = np.mean(episode_rewards[-eval_freq:])
            print(
                f"Episode {episode + 1:4d}: "
                f"training reward mean = {recent_average:8.2f}, "
                f"epsilon = {agent.epsilon:.3f}, "
                f"greedy evaluation mean = {evaluation['avg_reward']:8.2f}, "
                f"steps = {episode_length:2d}"
            )

    return {
        "episode_rewards": episode_rewards,
        "episode_lengths": episode_lengths,
        "evaluation_results": evaluation_results,
        "epsilon_history": epsilon_history,
        "losses": agent.losses,
        "q_values": agent.q_values,
    }


training_results = train_agent(
    agent, env, num_episodes=1000, eval_freq=100
)
final_eval = agent.evaluate(env, num_episodes=20)

print("\nFinal greedy evaluation on the generated yard")
print(
    f"Mean episode reward: {final_eval['avg_reward']:.2f} "
    f"+/- {final_eval['std_reward']:.2f}"
)
print(f"Mean episode length: {final_eval['avg_steps']:.1f} steps")
print(
    "Mean final utilization variance: "
    f"{final_eval['avg_final_variance']:.4f}"
)
print(f"Highest reward among evaluated episodes: {max(final_eval['total_rewards']):.2f}")

## 6. Compare with a movement-cost greedy baseline

The baseline repeatedly chooses the feasible move with the lowest movement cost per TEU quantity product. It is a simple reference policy, not an optimal solution method. Both policies are evaluated on resets of the same generated initial yard. The comparison is descriptive: a result from this one simulated instance does not establish general superiority.

In [ ]:
def greedy_baseline_rl(
    env: YardHousekeepingEnvironment, num_episodes: int = 20
) -> Dict:
    """Evaluate a feasible-move baseline that minimizes immediate move cost."""
    total_rewards = []
    total_steps = []
    final_variances = []

    for _ in range(num_episodes):
        env.reset()
        episode_reward = 0.0
        steps = 0
        done = False
        info = {}

        while not done and steps < env.max_time_steps:
            valid_actions = env.get_valid_actions(env.current_state)
            move_actions = [
                index
                for index in valid_actions
                if env.action_space[index].quantity > 0
            ]

            if move_actions:
                action_index = min(
                    move_actions,
                    key=lambda index: (
                        env.current_state.movement_costs[
                            env.action_space[index].source_block - 1,
                            env.action_space[index].destination_block - 1,
                        ]
                        * env.action_space[index].quantity
                    ),
                )
            else:
                action_index = len(env.action_space) - 1

            _, reward, done, info = env.step(action_index)
            episode_reward += reward
            steps += 1

        total_rewards.append(episode_reward)
        total_steps.append(steps)
        final_variances.append(info["utilization_variance"])

    return {
        "avg_reward": float(np.mean(total_rewards)),
        "std_reward": float(np.std(total_rewards)),
        "avg_steps": float(np.mean(total_steps)),
        "avg_final_variance": float(np.mean(final_variances)),
        "total_rewards": total_rewards,
    }


greedy_results = greedy_baseline_rl(env, num_episodes=20)

comparison = pd.DataFrame(
    [
        {
            "Method": "DQN agent",
            "Mean reward": final_eval["avg_reward"],
            "Reward standard deviation": final_eval["std_reward"],
            "Mean steps": final_eval["avg_steps"],
            "Mean final utilization variance": final_eval[
                "avg_final_variance"
            ],
        },
        {
            "Method": "Movement-cost greedy baseline",
            "Mean reward": greedy_results["avg_reward"],
            "Reward standard deviation": greedy_results["std_reward"],
            "Mean steps": greedy_results["avg_steps"],
            "Mean final utilization variance": greedy_results[
                "avg_final_variance"
            ],
        },
    ]
)
display(comparison)

if not np.isclose(greedy_results["avg_reward"], 0.0):
    reward_difference_percent = (
        (final_eval["avg_reward"] - greedy_results["avg_reward"])
        / abs(greedy_results["avg_reward"])
        * 100
    )
    print(f"DQN reward difference relative to baseline: {reward_difference_percent:.1f}%")
else:
    print("Relative reward difference is undefined because the baseline mean is zero.")

if not np.isclose(greedy_results["avg_final_variance"], 0.0):
    variance_difference_percent = (
        1
        - final_eval["avg_final_variance"]
        / greedy_results["avg_final_variance"]
    ) * 100
    print(
        "Final utilization-variance reduction relative to baseline: "
        f"{variance_difference_percent:.1f}%"
    )
else:
    print("Relative variance difference is undefined because baseline variance is zero.")

## 7. Inspect training and evaluation results

A moving average smooths short-term episode variation; it does not alter the rewards used for learning. Epsilon and loss are shown separately because they answer different questions: epsilon records the exploration schedule, while loss records the minibatch prediction error. The final panel compares mean evaluation rewards and their episode-to-episode standard deviations.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
ax1, ax2, ax3, ax4 = axes.flat

rewards = training_results["episode_rewards"]
episodes = np.arange(1, len(rewards) + 1)
ax1.plot(episodes, rewards, color="tab:blue", alpha=0.3, label="Episode reward")
window_size = 50
if len(rewards) >= window_size:
    moving_average = np.convolve(
        rewards, np.ones(window_size) / window_size, mode="valid"
    )
    ax1.plot(
        np.arange(window_size, len(rewards) + 1),
        moving_average,
        color="tab:red",
        linewidth=2,
        label=f"{window_size}-episode moving average",
    )
ax1.set_xlabel("Episode")
ax1.set_ylabel("Reward (dimensionless score)")
ax1.set_title("Training episode rewards")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(
    episodes,
    training_results["epsilon_history"],
    color="tab:green",
)
ax2.set_xlabel("Episode")
ax2.set_ylabel("Exploration probability, epsilon")
ax2.set_title("Epsilon-greedy exploration schedule")
ax2.set_ylim(0, 1.05)
ax2.grid(True, alpha=0.3)

losses = training_results["losses"]
if losses:
    loss_steps = np.arange(1, len(losses) + 1)
    ax3.plot(
        loss_steps[::10],
        losses[::10],
        color="tab:orange",
        alpha=0.7,
        label="Every tenth recorded update",
    )
    if len(losses) >= 100:
        loss_window = 100
        loss_average = np.convolve(
            losses, np.ones(loss_window) / loss_window, mode="valid"
        )
        ax3.plot(
            np.arange(loss_window, len(losses) + 1),
            loss_average,
            color="tab:red",
            linewidth=2,
            label=f"{loss_window}-update moving average",
        )
        ax3.legend()
else:
    ax3.text(
        0.5,
        0.5,
        "No loss values were recorded.",
        ha="center",
        va="center",
        transform=ax3.transAxes,
    )
ax3.set_xlabel("Training update")
ax3.set_ylabel("Mean squared Bellman error")
ax3.set_title("DQN training loss")
ax3.grid(True, alpha=0.3)

methods = ["DQN agent", "Movement-cost\ngreedy baseline"]
mean_rewards = [final_eval["avg_reward"], greedy_results["avg_reward"]]
std_rewards = [final_eval["std_reward"], greedy_results["std_reward"]]
positions = np.arange(len(methods))
ax4.bar(
    positions,
    mean_rewards,
    yerr=std_rewards,
    color=["tab:blue", "tab:red"],
    alpha=0.75,
    capsize=5,
)
ax4.set_xticks(positions)
ax4.set_xticklabels(methods)
ax4.set_ylabel("Mean episode reward (dimensionless score)")
ax4.set_title("Greedy evaluation comparison; error bars show standard deviation")
ax4.grid(True, axis="y", alpha=0.3)

fig.tight_layout()
plt.show()

## 8. Analyze the learned action choices

The policy analysis runs greedy episodes and records selected actions, observed rewards, Q-values, and utilization gaps at the time each action is chosen. Repeated choices describe behavior on this yard and under this learned network; they do not establish that a choice is optimal or that the agent has learned a general terminal strategy. No-op selections are included in the total action count but excluded from the move-preference table.

In [ ]:
def analyze_learned_policy(
    agent: DQNAgent, env: YardHousekeepingEnvironment
) -> Dict:
    """Collect greedy action frequencies and decision-time context."""
    num_test_episodes = 10
    action_counts = defaultdict(int)
    reward_by_action = defaultdict(list)
    state_action_pairs = []
    previous_epsilon = agent.epsilon
    agent.epsilon = 0.0

    try:
        for _ in range(num_test_episodes):
            state = env.reset()
            done = False
            steps = 0

            while not done and steps < env.max_time_steps:
                valid_actions = env.get_valid_actions(env.current_state)
                state_tensor = torch.as_tensor(
                    state, dtype=torch.float32, device=agent.device
                ).unsqueeze(0)
                was_training = agent.q_network.training
                agent.q_network.eval()
                with torch.no_grad():
                    q_values = agent.q_network(state_tensor).squeeze(0)
                if was_training:
                    agent.q_network.train()

                valid_tensor = torch.as_tensor(
                    valid_actions, dtype=torch.long, device=agent.device
                )
                best_position = torch.argmax(q_values[valid_tensor]).item()
                action_index = valid_actions[best_position]
                action = env.action_space[action_index]
                source_gap = (
                    env.current_state.get_block_by_id(
                        action.source_block
                    ).utilization_gap
                    if action.quantity > 0
                    else np.nan
                )
                destination_gap = (
                    env.current_state.get_block_by_id(
                        action.destination_block
                    ).utilization_gap
                    if action.quantity > 0
                    else np.nan
                )

                next_state, reward, done, info = env.step(action_index)
                action_counts[action_index] += 1
                reward_by_action[action_index].append(reward)
                if len(state_action_pairs) < 100:
                    state_action_record = {
                        "state": state.copy(),
                        "action": action_index,
                        "q_value": float(q_values[action_index].item()),
                        "reward": reward,
                        "utilization_variance": info[
                            "utilization_variance"
                        ],
                    }
                    if action.quantity > 0:
                        state_action_record["source_utilization_gap"] = source_gap
                        state_action_record[
                            "destination_utilization_gap"
                        ] = destination_gap
                    state_action_pairs.append(state_action_record)
                state = next_state
                steps += 1
    finally:
        agent.epsilon = previous_epsilon

    total_actions = sum(action_counts.values())
    action_preferences = {}
    for action_index, count in action_counts.items():
        action = env.action_space[action_index]
        if action.quantity == 0:
            continue
        action_preferences[action_index] = {
            "action": action,
            "count": count,
            "frequency": count / total_actions if total_actions else 0.0,
            "avg_reward": float(np.mean(reward_by_action[action_index])),
        }

    return {
        "action_preferences": action_preferences,
        "state_action_pairs": state_action_pairs,
        "total_actions": total_actions,
    }


policy_analysis = analyze_learned_policy(agent, env)
ranked_actions = sorted(
    policy_analysis["action_preferences"].items(),
    key=lambda item: item[1]["frequency"],
    reverse=True,
)

print(f"Actions observed, including no-op: {policy_analysis['total_actions']}")
print(f"Distinct move types selected: {len(ranked_actions)}")
print("\nMost frequently selected move types")
print(f"{'Rank':<6}{'Move':<32}{'Frequency':>12}{'Mean reward':>15}")
print("-" * 65)
for rank, (_, data) in enumerate(ranked_actions[:10], start=1):
    action = data["action"]
    label = (
        f"Block {action.source_block} to Block "
        f"{action.destination_block} ({action.quantity} TEU)"
    )
    print(
        f"{rank:<6}{label:<32}{data['frequency']:>11.1%}"
        f"{data['avg_reward']:>15.3f}"
    )

move_count = sum(data["count"] for _, data in ranked_actions)
positive_mean_reward_count = sum(
    data["count"] for _, data in ranked_actions if data["avg_reward"] > 0
)
if move_count:
    print(
        "Moves whose observed per-type mean reward is positive: "
        f"{positive_mean_reward_count / move_count:.1%} of move selections"
    )
else:
    print("No block-to-block moves were selected in the analysis episodes.")

## 9. Evaluate specified scenario variations

This section evaluates the same learned policy in altered copies of the generated yard. Poor weather changes the weather feature and movement-cost reward weight; high congestion changes congestion features and their reward weight; equipment failure changes recorded equipment availability. In this model, these attributes do not independently change move feasibility or movement-cost entries, so the results are limited scenario checks rather than evidence of operational robustness or generalization.

In [ ]:
def robustness_test(
    agent: DQNAgent, base_env: YardHousekeepingEnvironment
) -> Dict:
    """Evaluate the greedy policy on altered copies of the initial yard."""
    scenarios = [
        ("Normal", None),
        ("Poor Weather", 0.7),
        ("High Congestion", 1.0),
        ("Equipment Failure", 1.0),
    ]
    results = {}

    for scenario_name, weather_multiplier in scenarios:
        scenario_state = base_env.initial_state.copy()
        scenario_weights = base_env.reward_weights.copy()

        if scenario_name == "Poor Weather":
            scenario_state.weather_conditions = weather_multiplier
            scenario_weights["movement_cost"] *= 1.5
        elif scenario_name == "High Congestion":
            for block in scenario_state.blocks:
                block.congestion_level = min(1.0, block.congestion_level * 1.5)
            scenario_weights["congestion_penalty"] *= 2.0
        elif scenario_name == "Equipment Failure":
            for block in scenario_state.blocks:
                if random.random() < 0.3:
                    block.equipment_availability *= 0.5

        scenario_env = YardHousekeepingEnvironment(scenario_state)
        scenario_env.reward_weights = scenario_weights
        results[scenario_name] = agent.evaluate(
            scenario_env, num_episodes=10
        )
        print(
            f"{scenario_name}: mean reward = "
            f"{results[scenario_name]['avg_reward']:.2f}, "
            f"mean steps = {results[scenario_name]['avg_steps']:.1f}, "
            f"mean final utilization variance = "
            f"{results[scenario_name]['avg_final_variance']:.4f}"
        )

    return results


robustness_results = robustness_test(agent, env)
scenario_names = list(robustness_results)
scenario_rewards = [
    robustness_results[name]["avg_reward"] for name in scenario_names
]
scenario_variances = [
    robustness_results[name]["avg_final_variance"]
    for name in scenario_names
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
scenario_colors = plt.cm.Set3(np.linspace(0, 1, len(scenario_names)))
reward_bars = ax1.bar(
    scenario_names, scenario_rewards, color=scenario_colors, alpha=0.8
)
ax1.set_ylabel("Mean episode reward (dimensionless score)")
ax1.set_title("Greedy policy reward by scenario")
ax1.grid(True, axis="y", alpha=0.3)
ax1.tick_params(axis="x", rotation=30)
for bar, value in zip(reward_bars, scenario_rewards):
    ax1.annotate(
        f"{value:.2f}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        fontsize=9,
    )

variance_bars = ax2.bar(
    scenario_names, scenario_variances, color=scenario_colors, alpha=0.8
)
ax2.set_ylabel("Mean final utilization variance")
ax2.set_title("Final utilization variance by scenario")
ax2.grid(True, axis="y", alpha=0.3)
ax2.tick_params(axis="x", rotation=30)
for bar, value in zip(variance_bars, scenario_variances):
    ax2.annotate(
        f"{value:.4f}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        fontsize=9,
    )

fig.tight_layout()
plt.show()

## 10. Interpretation and limitations

The notebook demonstrates the mechanics of a DQN housekeeping agent: an encoded observation is mapped to Q-values, feasible actions are selected with an epsilon-greedy policy, rewards are calculated from a transition, and replayed transitions update the network using a masked Bellman target. The dueling heads, replay buffer, target network, and gradient clipping are components of this learning procedure.

Interpret the numerical outputs as measurements from the executed simulation, not as guaranteed properties of the method. Episode reward depends on reward scaling and weights; utilization variance measures dispersion among block utilizations but does not capture every operational objective. The greedy baseline is limited to immediate movement cost. The evaluation episodes reset to the same generated yard, so they do not constitute tests across independently generated yards.

No particular learning curve, best episode reward, percentage improvement, convergence claim, or emergent behavior is assumed. Such conclusions would require inspecting actual outputs and, for broad claims, additional experiments. The simulator also omits detailed stack accessibility, retrieval blocking, changing arrivals, and equipment-constrained transitions; these omissions limit how directly learned actions can be interpreted as terminal operating recommendations.